# 04 估计进阶

问题与建模思路参考 Allen B. Downey *Think Bayes*（中译《贝叶斯思维》）第 4 章。

**学习目标**：
- 用网格上的伯努利似然估计硬币正面概率 $x$
- 读后验的 MAP、均值与可信区间
- 用 Beta 共轭对照网格法，并观察「先验湮没」


In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
for candidate in [HERE, HERE / "notebooks" / "bayes", HERE.parent]:
    if (candidate / "thinkbayes_mini.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

from thinkbayes_mini import Suite

%matplotlib inline


def beta_pdf(x, alpha, beta):
    """Beta(α,β) density via log-gamma (no scipy)."""
    x = np.asarray(x, dtype=float)
    # support (0,1); endpoints handled as 0 for α,β > 1
    out = np.zeros_like(x)
    mask = (x > 0) & (x < 1)
    log_b = math.lgamma(alpha) + math.lgamma(beta) - math.lgamma(alpha + beta)
    out[mask] = np.exp(
        (alpha - 1) * np.log(x[mask]) + (beta - 1) * np.log(1 - x[mask]) - log_b
    )
    return out


## 1. 欧元问题

比利时一欧元硬币旋转落地：250 次中 140 次正面、110 次反面。估计正面概率 $x\in[0,1]$。

在网格 $x=0,0.01,\ldots,1$ 上放先验；单次观测似然：

$$
P(H\mid x)=x,\qquad P(T\mid x)=1-x
$$


In [ ]:
class Euro(Suite):
    def Likelihood(self, data, hypo):
        """hypo: 正面概率 x（0–100 的整数百分数）；data: 'H' 或 'T'."""
        x = hypo / 100
        return x if data == "H" else 1 - x


def make_uniform_euro() -> Euro:
    suite = Euro(range(0, 101))
    suite.Normalize()
    return suite


# 经典设定：250 次旋转，140 次正面、110 次反面
dataset = ["H"] * 140 + ["T"] * 110

euro = make_uniform_euro()
euro.UpdateSet(dataset)
print(f"MAP = {euro.MaximumLikelihood()}%")
print(f"后验均值 = {euro.Mean():.2f}%")
print(f"90% CI = {euro.CredibleInterval(90)}")


## 2. 后验摘要与图像


In [ ]:
xs, ps = zip(*sorted(euro.Items()))
plt.figure(figsize=(7, 4))
plt.plot(xs, ps)
plt.axvline(euro.MaximumLikelihood(), color="C1", ls="--", label="MAP")
plt.axvline(euro.Mean(), color="C2", ls=":", label="mean")
lo, hi = euro.CredibleInterval(90)
plt.axvspan(lo, hi, alpha=0.15, color="C0", label="90% CI")
plt.xlabel("x (%)")
plt.ylabel("p(x | data)")
plt.title("欧元问题：均匀先验下的后验")
plt.legend()
plt.tight_layout()


## 3. Beta 分布（共轭）

对伯努利 / 二项观测，Beta 先验共轭：

$$
x \sim \mathrm{Beta}(\alpha,\beta)
\quad\xrightarrow{\;h\text{ 次正面},\; t\text{ 次反面}\;}
\quad
x \mid D \sim \mathrm{Beta}(\alpha+h,\,\beta+t)
$$

均匀先验对应 $\alpha=\beta=1$。均值 $(\alpha)/(\alpha+\beta)$。


In [ ]:
alpha0, beta0 = 1, 1
h, t = 140, 110
alpha, beta = alpha0 + h, beta0 + t

grid = np.linspace(0, 1, 101)
pmf_probs = np.array([euro.Prob(i) for i in range(101)])
# 离散质量 → 密度近似：每格宽 0.01
density_approx = pmf_probs / 0.01

plt.figure(figsize=(7, 4))
plt.plot(grid * 100, beta_pdf(grid, alpha, beta), label=f"Beta({alpha},{beta}) pdf")
plt.plot(range(101), density_approx, "--", label="grid posterior / 0.01")
plt.xlabel("x (%)")
plt.ylabel("density")
plt.title("网格法 vs Beta 共轭")
plt.legend()
plt.tight_layout()

beta_mean = alpha / (alpha + beta)
print(f"Beta 均值 = {beta_mean*100:.2f}%")
print(f"网格均值 = {euro.Mean():.2f}%")


## 4. 先验湮没（swamping the prior）

不同先验在数据很少时差很大；数据变多后，后验被似然主导而靠近。


In [ ]:
def triangle_prior() -> Euro:
    """0→50 线性升，50→100 线性降（偏向公平硬币）。"""
    suite = Euro()
    for x in range(0, 51):
        suite.Set(x, x)
    for x in range(51, 101):
        suite.Set(x, 100 - x)
    suite.Normalize()
    return suite


def summarize(name, suite):
    print(f"{name}: mean={suite.Mean():.1f}, MAP={suite.MaximumLikelihood()}, CI90={suite.CredibleInterval(90)}")


fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, n_pairs, title in [
    (axes[0], 5, "数据少：5 正 5 反"),
    (axes[1], 140, "数据多：140 正 110 反"),
]:
    data = ["H"] * n_pairs + ["T"] * (n_pairs if n_pairs == 5 else 110)
    if n_pairs == 140:
        data = ["H"] * 140 + ["T"] * 110
    u, tr = make_uniform_euro(), triangle_prior()
    u.UpdateSet(data)
    tr.UpdateSet(data)
    ax.plot(*zip(*sorted(u.Items())), label="uniform")
    ax.plot(*zip(*sorted(tr.Items())), label="triangle")
    ax.set_title(title)
    ax.set_xlabel("x (%)")
    ax.legend()
    summarize(f"  uniform/{title}", u)
    summarize(f"  triangle/{title}", tr)
axes[0].set_ylabel("p(x|data)")
plt.tight_layout()


## 小结

1. 连续参数可先离散网格化，再用 `Suite` 更新。
2. Beta–Bernoulli 共轭给出同一问题的解析后验，可作网格法校验。
3. 先验重要，但足够多的数据会「淹没」先验差异。
